이미지 파일 압축 풀기 - 실행 하기 전 압축파일(*.zip)을 올려 놓아야 합니다.

TensorFlow 실행시 Epoch 와 Learn_Rate는 CoLab에서 지정하며

PyTorch 는 Raspberry Pi dl.py에서 지정한 값으로 설정됩니다.


In [ ]:
TensorFlow_Epoch = 5
TensorFlow_Learn_Rate = 0.001
mode = input('Pytorch 는 "p" Tensosor Flow 는 "t"를 입력하고 Enter를 누르세요.')
print()
if mode == 'p' or mode =='P':
    print('선택한 딥러닝 프레임웍은 PyTorch 입니다.')
elif mode == 't' or mode =='T':
    print('선택한 딥러닝 프레임웍은 TensorFlow 입니다.')


라이브러리 가져오기(Import)

In [ ]:
import glob
import zipfile
import pickle
fileList = glob.glob('*.zip')
zip_ref = zipfile.ZipFile(fileList[0], 'r');
zip_ref.extractall()
zip_ref.close()
w = fileList[0].split('.'); project = w[-2]
print('프로젝트:',project)

# Library ----------------------------------------------------------------------
import os
import sys
import random
import fnmatch
import time
import numpy as np
import cv2                                                                      # Open Cv 영상처리 라이브러리
import matplotlib.pyplot as plt
# PyTorch ----------------------------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary as summary
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import mean_squared_error, r2_score
# TensorFlow -------------------------------------------------------------------
import tensorflow as tf
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Input
from tensorflow.keras.optimizers import Adam
#-------------------------------------------------------------------------------
print(f'Python Version:        {sys.version}')                                  # Python Version
print(f'OpenCV version:        {cv2.__version__}')                              # Open CV version
print(f'Numpy version:         {np.__version__}')                               # Numpy version
print(f'Matplotlib version:    {plt.matplotlib.__version__}')                   # Matplotlib version
print(f'Tensorflow Version:    {tf.__version__}')                               # Tensorflow version
print(f'Keras Version:         {tensorflow.keras.__version__}')                 # KERAS version
print(f'Pytorch version:       {torch.__version__}')                            # Pytorch version


이미지 데이터 조향각 추출

In [ ]:
# @title
with open(f'{project}/_{project}.pickle', 'rb') as f:                           # pickle 파일
    x_test_Image = pickle.load(f)                                               # 시험(Test) 이미지 파일 이름 리스트
    x_valid_Image = pickle.load(f)                                              # 평가(Valid) 이미지 파일 이름 리스트
    x_train_Image = pickle.load(f)                                              # 훈련(Train) 이미지 파일 이름 리스트
    p = pickle.load(f)                                                          # 파라메터 리스트

if mode == 'p' or mode == 'P':
    EPOCHS = p[0]; learnRate = p[1]                                             # dl.py에서 설정한 학습회수
    #EPOCHS = 300; learnRate = 0.001                                             # 0.0005/0.0010/0.0015/0.0020/0.0025
elif mode == 't' or mode == 'T':
    EPOCHS = TensorFlow_Epoch
    learnRate = TensorFlow_Learn_Rate

y_test_Angle=[]
y_valid_Angle=[]
y_train_Angle = []
for f in x_test_Image:                                                          # 시험용 이미지 파일 이름에서 각도값
    y_test_Angle.append(int(f[-7:-4]))
for f in x_valid_Image:                                                         # 검증용 이미지 파일 이름에서 각도값
    y_valid_Angle.append(int(f[-7:-4]))
for f in x_train_Image:                                                         # 훈련용 이미지 파일 이름에서 각도값
    y_train_Angle.append(int(f[-7:-4]))

print('학습 회수:',EPOCHS,)
print('학습 비율', learnRate)
print('시험 이미지 개수:',len(x_test_Image))
print('검증 이미지 개수:',len(x_valid_Image))
print('훈련 이미지 개수:',len(x_train_Image))



PyTorch Nvidia CNN 모델 딥러닝

In [ ]:
# PyTorch Nvidia CNN 모델 구성 -------------------------------------------------
class NvidiaModel(nn.Module):
    def __init__(self):
      super(NvidiaModel, self).__init__()

      # elu=Expenential Linear Unit, similar to leaky Relu
      # skipping 1st hiddel layer (nomralization layer), as we have normalized the data

      # Convolution Layers
      self.layer1 = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=24, kernel_size=(5, 5), stride=(2, 2)),
        nn.ELU(inplace=True),
        nn.Conv2d(in_channels=24, out_channels=36, kernel_size=(5, 5), stride=(2, 2)),
        nn.ELU(inplace=True),
        nn.Conv2d(in_channels=36, out_channels=48, kernel_size=(5, 5), stride=(2, 2)),
        nn.ELU(inplace=True),
        nn.Conv2d(in_channels=48, out_channels=64, kernel_size=(3, 3)),
        nn.ELU(inplace=True),
        nn.Dropout(0.2),
        nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3)),
        nn.ELU(inplace=True)
      )

      # Fully Connected Layers
      self.layer2 = nn.Sequential(
        # nn.Flatten(),
        nn.Dropout(0.2),
        nn.Linear(in_features=18 * 64, out_features=100),
        nn.ELU(inplace=True),
        nn.Linear(in_features=100, out_features=50),
        nn.ELU(inplace=True),
        nn.Linear(in_features=50, out_features=10),
        nn.ELU(inplace=True)
      )

      # Output Layer
      self.layer3 = nn.Sequential(
        nn.Linear(in_features=10, out_features=1)
      )

    def forward(self, x):
      x = self.layer1(x)
      x = x.view(x.shape[0], -1)
      x = self.layer2(x)
      x = self.layer3(x)

      return x
#-------------------------------------------------------------------------------
def imgPaths(fGroup):
    r = []
    for filename in fGroup:
        r.append(os.path.join(project, filename))                               # 파일 경로(path)를 화일 이름 앞에 부착하여 리스트에 추가
    return(r)

# 학습 데이터 생성 -------------------------------------------------------------
class CustomDataset(Dataset):
    def __init__(self, imageList, angleList):
        self.imageList = imgPaths(imageList)
        self.angleList = angleList

        for i in range(len(imageList)):
            image = cv2.imread(self.imageList[i])
            image = image / 255
            self.imageList[i] = image

    def __len__(self):
        return len(self.imageList)

    def __getitem__(self, index):
        images = torch.FloatTensor(self.imageList[index]).permute(2,0,1)
        angles = torch.FloatTensor([self.angleList[index]])
        return images, angles
#-------------------------------------------------------------------------------
class EarlyStopping:
    # 주어진 patience 이후로 validation loss가 개선되지 않으면 학습을 조기 중지
    def __init__(self, patience=7, verbose=False, delta=0, path=f'_{project}_model_check.pt'):
        #
        # patience (int): validation loss가 개선된 후 기다리는 기간               - Default: 7
        # verbose (bool): True일 경우 각 validation loss의 개선 사항 메세지 출력  - Default: False
        # delta (float): 개선되었다고 인정되는 monitered quantity의 최소 변화     - Default: 0
        # path (str): checkpoint저장 경로

        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf                                              # np.Inf -> np.inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'Early Stopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        # validation loss가 감소하면 모델을 저장
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

def pyTorchlearnProc():
  model = NvidiaModel().to(device)

  train_dataset = CustomDataset(x_train_Image, y_train_Angle)
  train_loader = DataLoader(train_dataset, batch_size=len(x_train_Image))

  valid_dataset = CustomDataset(x_valid_Image, y_valid_Angle)
  valid_loader = DataLoader(valid_dataset, batch_size=len(x_valid_Image))

  optimizer = optim.Adam(model.parameters(), lr=learnRate)
  criterion = nn.MSELoss().to(device)

  early_stopping = EarlyStopping(patience=30, verbose=True)

  train_losses = []
  valid_losses = []
  avg_train_losses = []
  avg_valid_losses = []
  for epoch in range(1, EPOCHS + 1):
      # 학습
      model.train()

      for batch_idx, (data, target) in enumerate(train_loader):
          data, target = data.to(device), target.to(device)
          optimizer.zero_grad()

          output = model(data)

          loss = criterion(output.to(torch.float32), target.to(torch.float32))
          loss.backward()
          optimizer.step()  #loss를 백오더 한 후

          train_losses.append(loss.item())

      # 검증
      model.eval()

      with torch.no_grad():
        for data, target in valid_loader:
            data, target = data.to(device), target.to(device)

            output = model(data)

            loss = criterion(output.to(torch.float32), target.to(torch.float32))

            valid_losses.append(loss.item())

      train_loss = np.average(train_losses)
      valid_loss = np.average(valid_losses)
      avg_train_losses.append(train_loss)
      avg_valid_losses.append(valid_loss)

      epoch_len = len(str(EPOCHS))

      print_msg = (f'[{epoch:>{epoch_len}}/{EPOCHS:>{epoch_len}}] ' +
                      f'train_loss: {train_loss:.6f} ' +
                      f'valid_loss: {valid_loss:.6f}')

      print(print_msg)

      train_losses = []
      valid_losses = []

      early_stopping(valid_loss, model)

      if early_stopping.early_stop:
          print("Early stopping")
          break

  return {'loss':avg_train_losses, 'val_loss':avg_valid_losses}

#-------------------------------------------------------------------------------
def resultShow():                                                               # 학습 완료후 통계자료를 보여줍니다.
  model = NvidiaModel().to(device)                                              # NvidiaModel 의 Summary를 표시합니다.
  summary(model,(3, 66, 200))

  test_dataset = CustomDataset(x_test_Image, y_test_Angle)
  test_loader = DataLoader(test_dataset, batch_size=len(x_test_Image))

  model = NvidiaModel().to(device)
  model.load_state_dict(torch.load(f'_{project}_model_check.pt'))               # 모델 파일
  model.eval()

  y_pred = []
  y_target = []

  with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            print('pred', output.to(device).cpu().numpy())                                                              # GPU Edition
            print('target', target.to(device).cpu().numpy())                                                            #
            for output_data, target_data in zip(output.to(device).cpu().numpy(), target.to(device).cpu().numpy()):      #
              y_pred.append(output_data)
              y_target.append(target_data)

  mse = mean_squared_error(y_target, y_pred)
  r2s = r2_score(y_target, y_pred)
  #-----------------------------------------------------------------------------
  print('\n표준 편차:', f'{mse:.2f}')
  print('회귀 결정 계수:', f'{r2s:.2%}')
  #-----------------------------------------------------------------------------
  fig, axes = plt.subplots(1, len(x_test_Image), figsize=(30, 2+3))
  xTImg = imgPaths(x_test_Image)                                                # 파일 이름 앞에 경로(path) 추가
  for x, c in enumerate(xTImg):
    yuv_image = cv2.imread(c, cv2.IMREAD_UNCHANGED)                             # 테스트 이미지 읽기
    bgr_image = cv2.cvtColor(yuv_image, cv2.COLOR_YUV2BGR)                      # YUV 이미지를 BGR 형식으로 변환
    rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)                      # MatPlot Lib 에서 표시되는 RGB 형식으로 변환
    axes[x].imshow(rgb_image)
    t = x_test_Image[x][-7:-4] +'/'+str(int(y_pred[x]))
    axes[x].set_title(t, fontsize=60)
    axes[x].xaxis.set_ticks([])
    axes[x].yaxis.set_ticks([])


TensorFlow Nvidia CNN 모델 딥러닝

In [ ]:
#TensorFlow NVIDIA CNN 모델 구성
def nvidia_model(lr=learnRate):

    model = Sequential(name='Nvidia_Model')

    # elu=Expenential Linear Unit, similar to leaky Relu
    # skipping 1st hiddel layer (nomralization layer), as we have normalized the data
    # Convolution Layers

    model.add(Input(shape=(66, 200, 3)))
    model.add(Conv2D(24, (5, 5), strides=(2, 2), activation='elu'))
    model.add(Conv2D(36, (5, 5), strides=(2, 2), activation='elu'))
    model.add(Conv2D(48, (5, 5), strides=(2, 2), activation='elu'))
    model.add(Conv2D(64, (3, 3), activation='elu'))
    model.add(Conv2D(64, (3, 3), activation='elu'))

    # Fully Connected Layers
    model.add(Flatten())
    model.add(Dropout(0.2))
    model.add(Dense(100, activation='elu'))
    model.add(Dense(50, activation='elu'))
    model.add(Dense(10, activation='elu'))
    model.add(Dense(1)) # output layer: turn angle (from -99 ~ +99  0 is straight, -99 turn left, +99 turn right)

    # since this is a regression problem not classification problem,
    # we use MSE (Mean Squared Error) as loss function
    # optimizer = Adam(lr=1e-3)  # lr is learning rate
    optimizer = Adam(learning_rate=lr)  # 학습율 설정
    model.compile(loss='mse', optimizer=optimizer)

    return model

# 이미지 읽기, 정규화 ----------------------------------------------------------
def batchGen(imageList, angleList, batchSize):

    while True:
        imageBatch = []
        angleBatch = []

        for i in range(batchSize):
            t = f'{project}/'+ imageList[i]
            image = cv2.imread(t)
            angle = angleList[i]

            image = image/255                                                   # 255 로 나누어 0 - 0.999 로 데이터 표시 방법 변경
            imageBatch.append(image)
            angleBatch.append(angle)

        yield(np.asarray(imageBatch), np.asarray(angleBatch))
# 모델 학습 --------------------------------------------------------------------
def tensorFlowLearnProc(model, x_train_Image, y_train_Angle, x_valid_Image, y_valid_Angle, EPOCHS, project):

    startLearn = time.time()   #  학습 시작 시각
    cPCallback = tensorflow.keras.callbacks.ModelCheckpoint(filepath=f'./_{project}_model_check.keras', verbose=1, save_best_only=True)
    hist = model.fit(batchGen(x_train_Image, y_train_Angle, batchSize=len(x_train_Image)),
                              steps_per_epoch=100,
                              epochs=EPOCHS,                                    # 학습 회수
                              validation_data = batchGen(x_valid_Image, y_valid_Angle, batchSize=len(x_valid_Image)),
                              validation_steps=200,
                              verbose=1,                                        # 막대 그래프로 진행상황 표시
                              shuffle=True,                                     # 주어진 데이터를 랜덤하게 재배치.
                              callbacks=[cPCallback])

    # TensorFlow Lite 변환기 생성
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # 양자화 (선택 사항)
    #converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # 변환 및 저장
    tflite_model = converter.convert()

    # TensorFlow Lite 모델 저장
    open(f'./_{project}_model_final.tflite', 'wb').write(tflite_model)
    model.save(f'./_{project}_model_final.keras')   # 학습모델 저장

    endLearn = time.time()                   # 학습 종료 시각
    timeLearn = int(endLearn - startLearn)   # 학습 소요 시간

    t = f'{int(timeLearn/60):4d} 분   {int(timeLearn%60):2d} 초'
    print('학습 소요시간:',t)

    return(hist.history)   # 딕셔너리 데이터로 변환하여 반환

# 테스트 이미지 예측각도 -------------------------------------------------------
def testImagesPredict(x_test_Image, project):
  # TensorFlow Prediction - Load the trained model
  model = tf.keras.models.load_model(f'./_{project}_model_final.keras')

  # Function to preprocess the test image
  y_pred = []

  for i in range(len(x_test_Image)):
    # Choose a test image path
    test_image_path = f'{project}/' + x_test_Image[i]  # Replace with your test image path

    image = cv2.imread(test_image_path)
    test_image = image / 255.0  # Normalize the image

    # Reshape the image for model input (if necessary)
    test_image = np.expand_dims(test_image, axis=0)

    # Make a prediction
    predicted_angle = model.predict(test_image, verbose=0)[0][0]
    y_pred.append(int(predicted_angle))

    # Print the predicted angle
    print(f"Predicted angle for {test_image_path}: {predicted_angle}")

  return(y_pred)

#-------------------------------------------------------------------------------
def pltPlot(history, project):
    min_loss = min(history['loss'])
    min_val_loss = min(history['val_loss'])

    plt.xticks(np.arange(len(history['loss'])), np.arange(1, len(history['loss']) + 1))
    plt.plot(history['loss'], color='blue')
    plt.plot(history['val_loss'],color='red')
    plt.legend([f'Training Loss  = {min_loss:.3f}', f'Validation Loss = {min_val_loss:.3f}'])
    plt.title(f'{project}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.savefig(f'./_{project}_Plt.png')
    plt.show()



학습 결과 표시, 테스트 파일 각도 예측

In [ ]:
startTime = time.time()                                                         # 학습 시작 시간 저장

if mode == 'p' or mode == 'P':
    # PyTorch 학습 -------------------------------------------------------------
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f'{device} is available')
    history = pyTorchlearnProc()                                                           # 학습 시작
    lossMin = min(history['loss'])                                                  # 딕셔너리 history의 'loss'키 데이터는 리스트, 최소값
    val_lossMin = min(history['val_loss'])                                          # 딕셔너리 history의 'val_loss'키 데이터는 리스트, 최소값
    #---------------------------------------------------------------------------
    plt.plot(history['loss'],color='blue')
    plt.plot(history['val_loss'],color='red')
    plt.legend([f'training loss Min: {lossMin:0.2f}', f'validation loss Min: {val_lossMin:0.2f}'])
    plt.savefig(f'_{project}_result.png')

    if len(x_test_Image): resultShow()

elif mode == 't' or mode == 'T':
    # TensorFlow 학습 ----------------------------------------------------------
    model = nvidia_model()                                                          # NVIDIA 모델
    model.summary()                                                                 # 모델 요약
    history = tensorFlowLearnProc(model, x_train_Image, y_train_Angle, x_valid_Image, y_valid_Angle, EPOCHS, project)
    pltPlot(history, project)                                                       # Loss 그래프 출력
    angleList = testImagesPredict(x_test_Image, project)                            # 테스트 이미지 각도 예측

    # 테스트 이미지 라벨 조향각/예측 조향각 ------------------------------------
    xTImg = []
    for filename in x_test_Image:
        xTImg.append(os.path.join(project, filename))                             # 파일 경로(path)를 화일 이름 앞에 부착하여 리스트에 추가

    fig, axes = plt.subplots(1, len(x_test_Image), figsize=(30, 2+3))

    for x, c in enumerate(xTImg):
        yuv_image = cv2.imread(c, cv2.IMREAD_UNCHANGED)                               # 테스트 이미지 읽기
        bgr_image = cv2.cvtColor(yuv_image, cv2.COLOR_YUV2BGR)                        # YUV 이미지를 BGR 형식으로 변환
        rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)                        # MatPlot Lib 에서 표시되는 RGB 형식으로 변환
        axes[x].imshow(rgb_image)
        t = x_test_Image[x][-7:-4] + '/' + str(int(angleList[x]))

        axes[x].set_title(t, fontsize=50)
        axes[x].xaxis.set_ticks([])
        axes[x].yaxis.set_ticks([])
    plt.show()

#-------------------------------------------------------------------------------
elapsedTime = int(time.time() - startTime)                                      # 학습 경과 시간 저장
print("학습 소요 시간:", int(elapsedTime/60), '분', elapsedTime%60, '초' )      # 학습 소요시간 프린트

